In [1]:
# Import all required libraries
import time
import numpy as np
from pyspark import SparkContext, SparkConf

# Initialize Spark
conf = SparkConf().setAppName("CSL7110_Assignment4").setMaster("local[*]")
sc = SparkContext(conf=conf)

print("PySpark version:", sc.version)
print("Spark initialized successfully!")
     

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 15:09:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark version: 4.1.1
Spark initialized successfully!


In [2]:
from pyspark import SparkContext

# Stop existing Spark context if running
try:
    sc.stop()
except:
    pass

# Start new Spark context
sc = SparkContext.getOrCreate()

print("Spark initialized successfully!")
print("Spark version:", sc.version)

Spark initialized successfully!
Spark version: 4.1.1


In [4]:
def load_graph(filename):
    # Read the file
    lines = sc.textFile(filename)

    # Parse each line as (source, destination)
    edges = lines.map(lambda line: (int(line.split()[0]), int(line.split()[1])))

    # Remove duplicate edges
    edges = edges.distinct()

    # Build adjacency list: (source, [list of neighbors])
    adjacency = edges.groupByKey().mapValues(list)

    return adjacency, edges

# Test with small graph first
print("Loading small.txt...")
adj_small, edges_small = load_graph('CSL7110_Assignment4/small.txt')

# Count nodes and edges
num_nodes_small = adj_small.count()
num_edges_small = edges_small.count()

print(f"Small graph loaded!")
print(f"Number of nodes: {num_nodes_small}")
print(f"Number of edges: {num_edges_small}")

Loading small.txt...
Small graph loaded!
Number of nodes: 100
Number of edges: 950


In [5]:
def pagerank(graph_file, num_iterations=40, beta=0.8):
    # Load graph
    adjacency, edges = load_graph(graph_file)

    # Get all unique nodes
    all_nodes = edges.flatMap(lambda x: [x[0], x[1]]).distinct()
    n = all_nodes.count()

    print(f"Total nodes: {n}")
    print(f"Running PageRank for {num_iterations} iterations with β={beta}")

    # Initialize ranks equally to 1/n
    ranks = all_nodes.map(lambda node: (node, 1.0 / n))

    # Run iterations
    for i in range(num_iterations):
        # Join ranks with adjacency list
        contributions = adjacency.join(ranks).flatMap(
            lambda x: [
                (neighbor, x[1][1] / len(x[1][0]))
                for neighbor in x[1][0]
            ]
        )

        # Add teleport probability and sum contributions
        ranks = contributions.reduceByKey(lambda a, b: a + b).mapValues(
            lambda rank: (1 - beta) / n + beta * rank
        )

        if (i + 1) % 10 == 0:
            print(f"Iteration {i+1} completed")

    return ranks, n

In [6]:
# Run on small graph first
print("="*50)
print("Running PageRank on small.txt")
print("="*50)
ranks_small, n_small = pagerank('CSL7110_Assignment4/small.txt')

# Get top 5 and bottom 5
ranks_list_small = ranks_small.collect()
ranks_sorted_small = sorted(ranks_list_small, key=lambda x: x[1], reverse=True)

print("\nTop 5 nodes:")
for node, score in ranks_sorted_small[:5]:
    print(f"  Node {node}: {score:.6f}")

print("\nBottom 5 nodes:")
for node, score in ranks_sorted_small[-5:]:
    print(f"  Node {node}: {score:.6f}")

print(f"\nTop score: {ranks_sorted_small[0][1]:.6f}")
print("Expected top score: ~0.036")

Running PageRank on small.txt
Total nodes: 100
Running PageRank for 40 iterations with β=0.8
Iteration 10 completed
Iteration 20 completed
Iteration 30 completed
Iteration 40 completed

Top 5 nodes:
  Node 53: 0.035731
  Node 14: 0.034171
  Node 40: 0.033630
  Node 1: 0.030006
  Node 27: 0.029720

Bottom 5 nodes:
  Node 89: 0.003922
  Node 37: 0.003808
  Node 81: 0.003695
  Node 59: 0.003670
  Node 85: 0.003410

Top score: 0.035731
Expected top score: ~0.036


In [7]:
#Step 3: Run PageRank on Full Graph (whole.txt)
# Run on full graph
print("="*50)
print("Running PageRank on whole.txt")
print("="*50)

ranks_whole, n_whole = pagerank('CSL7110_Assignment4/whole.txt')

# Get top 5 and bottom 5
ranks_list_whole = ranks_whole.collect()
ranks_sorted_whole = sorted(ranks_list_whole, key=lambda x: x[1], reverse=True)

print("\nTop 5 nodes with highest PageRank scores:")
for node, score in ranks_sorted_whole[:5]:
    print(f"  Node {node}: {score:.6f}")

print("\nBottom 5 nodes with lowest PageRank scores:")
for node, score in ranks_sorted_whole[-5:]:
    print(f"  Node {node}: {score:.6f}")

Running PageRank on whole.txt
Total nodes: 1000
Running PageRank for 40 iterations with β=0.8
Iteration 10 completed
Iteration 20 completed
Iteration 30 completed
Iteration 40 completed


[Stage 177:==============================================>       (71 + 11) / 82]


Top 5 nodes with highest PageRank scores:
  Node 263: 0.002020
  Node 537: 0.001943
  Node 965: 0.001925
  Node 243: 0.001853
  Node 285: 0.001827

Bottom 5 nodes with lowest PageRank scores:
  Node 408: 0.000388
  Node 424: 0.000355
  Node 62: 0.000353
  Node 93: 0.000351
  Node 558: 0.000329
